# 01 — VocaLip Thai: Supervised Word-Level Baseline

**Experiment 0 — Baseline for the VocaLip Thai research pipeline**

Research direction:

`video → visual preprocessing → representation learning → temporal modeling/alignment → Thai VSR → LLM-assisted decoding`

This notebook intentionally starts with a **simple supervised closed-vocabulary baseline** before adding the proposed advanced components.

## Current task

- 44 Thai words
- Mouth-only baseline
- 16 frames per word clip
- 112×112 RGB mouth ROI
- CNN frame encoder
- BiGRU temporal encoder
- Temporal attention
- 44-class classifier
- CrossEntropyLoss
- Accuracy + Macro F1 + Top-k + Confusion Matrix

## Not included yet

- Self-supervised pretraining
- Explicit temporal alignment loss
- Face + Mouth dual-stream fusion
- LLM-assisted decoding
- Continuous phrase/sentence recognition

These will be developed in later notebooks so each contribution can be measured separately.


## VocaLip Thai Experimental Roadmap

This folder is organized as a progressive research pipeline:

1. **01 — Supervised Word-Level Baseline** ← current notebook
2. **02 — Self-Supervised Visual Pretraining**
3. **03 — SSL Fine-Tuning for 44 Thai Words**
4. **04 — Temporal Modeling / Alignment Ablation**
5. **05 — Face + Mouth Dual-Stream Ablation**
6. **06 — LLM-Assisted Decoding / Rescoring**
7. **07 — Real-Time Inference + Prototype Evaluation**

The baseline is kept intentionally simple so later improvements can be evaluated with a clear ablation study.


In [ ]:
# CELL 1 : ENVIRONMENT / IMPORTS

import os
import sys
import json
import math
import random
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", DEVICE)

In [ ]:
# CELL 2 : CONFIG

ROOT = Path(r"D:\Face_detection\dataset\word_level_v5\video001")

MANIFEST_PATH = ROOT / "word_train_manifest_v5_training_ready.csv"
LABEL_MAP_PATH = ROOT / "word_label_map_v5.json"

CACHE_ROOT = ROOT / "mouth_cache_v3"

NUM_CLASSES = 44
TARGET_FRAMES = 16
ROI_SIZE = 112

BATCH_SIZE = 16
NUM_WORKERS = 0          # Windows/Jupyter safe baseline
LR = 3e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 40
PATIENCE = 8

SEED = 42

# Imbalance strategy:
# Start with weighted CrossEntropyLoss.
# Do NOT use WeightedRandomSampler at the same time unless testing intentionally.
USE_CLASS_WEIGHTED_LOSS = True
USE_WEIGHTED_SAMPLER = False

MODEL_DIR = ROOT / "models_01_baseline"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "01_word_level_baseline_best.pt"

print("Manifest :", MANIFEST_PATH)
print("Label map:", LABEL_MAP_PATH)
print("Cache    :", CACHE_ROOT)
print("Model dir:", MODEL_DIR)

In [ ]:
# CELL 3 : REPRODUCIBILITY

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Reproducibility first. Can be relaxed later for speed.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("Seed:", SEED)

In [ ]:
# CELL 4 : LOAD FINAL WORD-LEVEL DATASET

assert MANIFEST_PATH.exists(), f"Manifest not found: {MANIFEST_PATH}"
assert LABEL_MAP_PATH.exists(), f"Label map not found: {LABEL_MAP_PATH}"

df = pd.read_csv(MANIFEST_PATH, encoding="utf-8-sig")

with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    label_map = json.load(f)

required_cols = {
    "split",
    "word",
    "model_class_id",
    "video_clip_path"
}

missing = required_cols - set(df.columns)
assert not missing, f"Missing columns: {missing}"

df["model_class_id"] = df["model_class_id"].astype(int)

print("Samples :", len(df))
print("Classes :", df["model_class_id"].nunique())
print("Words   :", df["word"].nunique())
print()
print(df["split"].value_counts())

assert df["model_class_id"].nunique() == NUM_CLASSES
assert df["model_class_id"].min() == 0
assert df["model_class_id"].max() == NUM_CLASSES - 1

display(df.head())

In [ ]:
# CELL 5 : DATASET AUDIT — DO NOT RESPLIT

split_summary = (
    df.groupby("split")
      .agg(
          samples=("model_class_id", "size"),
          classes=("model_class_id", "nunique"),
          words=("word", "nunique"),
      )
)

print(split_summary)

for split in ["train", "val", "test"]:
    sub = df[df["split"] == split]
    counts = sub["model_class_id"].value_counts()

    print(
        f"{split:5s} | samples={len(sub):4d} | "
        f"classes={sub['model_class_id'].nunique():2d} | "
        f"min/class={counts.min():2d} | "
        f"max/class={counts.max():3d} | "
        f"mean/class={counts.mean():.2f}"
    )

# Existing split is preserved by design.
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

assert train_df["model_class_id"].nunique() == NUM_CLASSES
assert val_df["model_class_id"].nunique() == NUM_CLASSES
assert test_df["model_class_id"].nunique() == NUM_CLASSES

print("\n✅ Existing train/val/test split preserved")

## Mouth cache policy

Version 3 should **reuse cache if it already exists**.

Expected cache file per sample:

`CACHE_ROOT / split / <word_clip_id or clip stem>.npz`

Each `.npz` contains:

- `frames`: shape `(16, 112, 112, 3)`, uint8

The training loop never runs MediaPipe.  
Mouth extraction is a **one-time preprocessing stage** and can be replaced by any better cropper later without rewriting the model/training code.

In [ ]:
# CELL 6 : CACHE NAMING + CACHE AUDIT

def sample_key(row):
    if "word_clip_id" in row.index and pd.notna(row["word_clip_id"]):
        return str(row["word_clip_id"])
    return Path(str(row["video_clip_path"])).stem

def cache_path_for_row(row):
    return CACHE_ROOT / str(row["split"]) / f"{sample_key(row)}.npz"

df["mouth_cache_path"] = df.apply(lambda r: str(cache_path_for_row(r)), axis=1)

cache_exists = df["mouth_cache_path"].map(lambda p: Path(p).exists())
print(f"Existing cache: {cache_exists.sum():,}/{len(df):,} ({cache_exists.mean()*100:.2f}%)")

print("\nBy split:")
for split in ["train", "val", "test"]:
    m = df["split"].eq(split)
    hit = cache_exists[m].sum()
    total = m.sum()
    print(f"{split:5s}: {hit:4d}/{total:4d}")

if cache_exists.all():
    print("\n✅ All mouth caches already exist. Skip preprocessing.")
else:
    print("\n⚠️ Cache incomplete.")
    print("Do NOT train yet. Build only the missing cache files using Cell 7.")

In [ ]:
# CELL 7 : OPTIONAL — BUILD MISSING MOUTH CACHE WITH MEDIAPIPE VIDEO MODE
# Run this cell only if Cell 6 reports missing caches.
#
# Requirements:
#   pip install mediapipe
# Face landmarker model:
#   D:\Face_detection\models\face_landmarker.task

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

FACE_LANDMARKER_MODEL = Path(r"D:\Face_detection\models\face_landmarker.task")

LIP_INDICES = [
    61, 146, 91, 181, 84, 17, 314, 405,
    321, 375, 291, 308, 324, 318, 402,
    317, 14, 87, 178, 88, 95, 185, 40,
    39, 37, 0, 267, 269, 270, 409,
    415, 310, 311, 312, 13, 82, 81,
    80, 191, 78
]

FACE_DETECTION_CONF = 0.30
FACE_PRESENCE_CONF = 0.30
TRACKING_CONF = 0.30
MOUTH_SCALE = 2.2
SMOOTH_ALPHA = 0.30

assert FACE_LANDMARKER_MODEL.exists(), FACE_LANDMARKER_MODEL

def create_landmarker():
    base_options = python.BaseOptions(model_asset_path=str(FACE_LANDMARKER_MODEL))
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1,
        min_face_detection_confidence=FACE_DETECTION_CONF,
        min_face_presence_confidence=FACE_PRESENCE_CONF,
        min_tracking_confidence=TRACKING_CONF,
    )
    return vision.FaceLandmarker.create_from_options(options)

def temporal_resample(frames, target_frames=16):
    if len(frames) == 0:
        raise ValueError("No frames")
    if len(frames) == target_frames:
        return np.stack(frames)
    idx = np.linspace(0, len(frames)-1, target_frames)
    idx = np.round(idx).astype(int)
    return np.stack([frames[i] for i in idx])

def interp_1d(values):
    values = np.asarray(values, dtype=np.float32)
    valid = np.isfinite(values)
    if not valid.any():
        return values
    x = np.arange(len(values))
    values[~valid] = np.interp(x[~valid], x[valid], values[valid])
    return values

def ema(values, alpha=0.30):
    out = values.copy()
    for i in range(1, len(out)):
        out[i] = alpha * out[i] + (1-alpha) * out[i-1]
    return out

def extract_mouth_sequence(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0:
        fps = 30.0

    frames_bgr = []
    boxes = []

    landmarker = create_landmarker()
    try:
        frame_idx = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            frames_bgr.append(frame)

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb = np.ascontiguousarray(rgb)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

            timestamp_ms = int(round(frame_idx * 1000.0 / fps))
            result = landmarker.detect_for_video(mp_img, timestamp_ms)

            if result.face_landmarks:
                lm = result.face_landmarks[0]
                h, w = frame.shape[:2]

                xs = np.array([lm[i].x * w for i in LIP_INDICES])
                ys = np.array([lm[i].y * h for i in LIP_INDICES])

                x1, x2 = xs.min(), xs.max()
                y1, y2 = ys.min(), ys.max()

                cx = (x1 + x2) / 2
                cy = (y1 + y2) / 2

                side = max(x2-x1, y2-y1) * MOUTH_SCALE
                boxes.append((cx, cy, side))
            else:
                boxes.append((np.nan, np.nan, np.nan))

            frame_idx += 1
    finally:
        landmarker.close()
        cap.release()

    if len(frames_bgr) == 0:
        raise RuntimeError("Video contains no frames")

    box_arr = np.asarray(boxes, dtype=np.float32)
    if np.isfinite(box_arr[:, 0]).sum() == 0:
        raise RuntimeError("ZERO_FACE_DETECTION")

    cx = ema(interp_1d(box_arr[:,0]), SMOOTH_ALPHA)
    cy = ema(interp_1d(box_arr[:,1]), SMOOTH_ALPHA)
    side = ema(interp_1d(box_arr[:,2]), SMOOTH_ALPHA)

    crops = []

    for frame, x, y, s in zip(frames_bgr, cx, cy, side):
        h, w = frame.shape[:2]

        x1 = int(round(x - s/2))
        y1 = int(round(y - s/2))
        x2 = int(round(x + s/2))
        y2 = int(round(y + s/2))

        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(w, x2)
        y2 = min(h, y2)

        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            raise RuntimeError("Empty mouth crop")

        crop = cv2.resize(crop, (ROI_SIZE, ROI_SIZE), interpolation=cv2.INTER_AREA)
        crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        crops.append(crop)

    seq = temporal_resample(crops, TARGET_FRAMES)
    return seq.astype(np.uint8)

def build_missing_cache(dataframe):
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)

    ok_count = 0
    fail_count = 0
    failures = []

    for i, row in dataframe.iterrows():
        out_path = cache_path_for_row(row)
        if out_path.exists():
            continue

        out_path.parent.mkdir(parents=True, exist_ok=True)

        try:
            seq = extract_mouth_sequence(row["video_clip_path"])
            assert seq.shape == (TARGET_FRAMES, ROI_SIZE, ROI_SIZE, 3)
            np.savez_compressed(out_path, frames=seq)
            ok_count += 1
        except Exception as e:
            fail_count += 1
            failures.append({
                "index": int(i),
                "word": row["word"],
                "video_clip_path": row["video_clip_path"],
                "error": repr(e),
            })

        if (ok_count + fail_count) % 50 == 0:
            print(f"processed={ok_count+fail_count:,} | new_cache={ok_count:,} | failed={fail_count:,}")

    fail_df = pd.DataFrame(failures)

    print("\nDONE")
    print("New cache :", ok_count)
    print("Failed    :", fail_count)

    if len(fail_df):
        fail_path = ROOT / "mouth_cache_v3_failures.csv"
        fail_df.to_csv(fail_path, index=False, encoding="utf-8-sig")
        print("Failure CSV:", fail_path)

    return fail_df

# Uncomment only when ready:
# failures = build_missing_cache(df)

In [ ]:
# CELL 8 : FINAL CACHE VALIDATION

df["mouth_cache_path"] = df.apply(lambda r: str(cache_path_for_row(r)), axis=1)

missing_cache = df[~df["mouth_cache_path"].map(lambda p: Path(p).exists())].copy()

print("Total samples :", len(df))
print("Missing cache :", len(missing_cache))

if len(missing_cache):
    display(missing_cache[["split", "word", "video_clip_path", "mouth_cache_path"]].head(20))
    raise RuntimeError("Cache is incomplete. Do not train yet.")

# Shape check on a sample
sample_cache = Path(df.iloc[0]["mouth_cache_path"])
arr = np.load(sample_cache)["frames"]

print("Sample cache:", sample_cache)
print("Shape       :", arr.shape)
print("dtype       :", arr.dtype)

assert arr.shape == (TARGET_FRAMES, ROI_SIZE, ROI_SIZE, 3)
print("\n✅ Mouth cache is training-ready")

In [ ]:
# CELL 9 : DATASET + AUGMENTATION

class LipWordDataset(Dataset):
    def __init__(self, dataframe, train=False):
        self.df = dataframe.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.df)

    def augment(self, x):
        # x: [T,C,H,W], float in [0,1]

        # Mild horizontal flip.
        # For word-level mouth classification this is usually acceptable,
        # but keep probability low and verify experimentally.
        if random.random() < 0.25:
            x = torch.flip(x, dims=[3])

        # Mild brightness jitter
        if random.random() < 0.30:
            gain = random.uniform(0.90, 1.10)
            x = torch.clamp(x * gain, 0.0, 1.0)

        # Mild temporal frame dropout by replacing one frame with neighbor
        if x.shape[0] > 2 and random.random() < 0.20:
            t = random.randint(1, x.shape[0]-2)
            x[t] = x[t-1]

        return x

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        data = np.load(row["mouth_cache_path"])
        frames = data["frames"]

        # [T,H,W,C] -> [T,C,H,W]
        x = torch.from_numpy(frames).float().permute(0, 3, 1, 2) / 255.0

        if self.train:
            x = self.augment(x)

        y = int(row["model_class_id"])

        return {
            "x": x,
            "y": torch.tensor(y, dtype=torch.long),
            "word": str(row["word"]),
            "path": str(row["video_clip_path"]),
        }

train_ds = LipWordDataset(train_df.assign(
    mouth_cache_path=train_df.apply(lambda r: str(cache_path_for_row(r)), axis=1)
), train=True)

val_ds = LipWordDataset(val_df.assign(
    mouth_cache_path=val_df.apply(lambda r: str(cache_path_for_row(r)), axis=1)
), train=False)

test_ds = LipWordDataset(test_df.assign(
    mouth_cache_path=test_df.apply(lambda r: str(cache_path_for_row(r)), axis=1)
), train=False)

sample = train_ds[0]
print("x:", sample["x"].shape, sample["x"].dtype)
print("y:", sample["y"].item())
print("word:", sample["word"])

In [ ]:
# CELL 10 : CLASS WEIGHTS + DATALOADERS

train_counts = (
    train_df["model_class_id"]
    .value_counts()
    .sort_index()
    .reindex(range(NUM_CLASSES), fill_value=0)
)

assert (train_counts > 0).all()

# Inverse-frequency with normalization
class_weights = len(train_df) / (NUM_CLASSES * train_counts.values.astype(np.float32))
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

print("Class weight range:", float(class_weights.min()), "->", float(class_weights.max()))

sampler = None

if USE_WEIGHTED_SAMPLER:
    sample_weights = [
        float(class_weights[int(y)].cpu())
        for y in train_df["model_class_id"].tolist()
    ]
    sampler = WeightedRandomSampler(
        sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=(sampler is None),
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

batch = next(iter(train_loader))
print("Batch x:", batch["x"].shape)
print("Batch y:", batch["y"].shape)

In [ ]:
# CELL 11 : MODEL — CNN FRAME ENCODER + BiGRU

class FrameEncoder(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 192, 3, stride=2, padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(192, out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.20),
        )

    def forward(self, x):
        return self.proj(self.features(x))


class MouthWordClassifier(nn.Module):
    def __init__(
        self,
        num_classes=44,
        frame_dim=256,
        hidden_dim=256,
        num_layers=2,
        dropout=0.30
    ):
        super().__init__()

        self.frame_encoder = FrameEncoder(frame_dim)

        self.temporal = nn.GRU(
            input_size=frame_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.temporal_attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_classes),
        )

    def forward(self, x):
        # x [B,T,C,H,W]
        B, T, C, H, W = x.shape

        z = x.reshape(B*T, C, H, W)
        z = self.frame_encoder(z)
        z = z.reshape(B, T, -1)

        z, _ = self.temporal(z)

        attn = self.temporal_attention(z).squeeze(-1)
        attn = torch.softmax(attn, dim=1)

        pooled = torch.sum(z * attn.unsqueeze(-1), dim=1)
        logits = self.classifier(pooled)

        return logits

model = MouthWordClassifier(num_classes=NUM_CLASSES).to(DEVICE)

x = batch["x"][:2].to(DEVICE)
with torch.no_grad():
    logits = model(x)

print(model.__class__.__name__)
print("Input :", x.shape)
print("Output:", logits.shape)

assert logits.shape == (len(x), NUM_CLASSES)

params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parameters: {params:,}")
print(f"Trainable : {trainable:,}")

In [ ]:
# CELL 12 : LOSS / OPTIMIZER / SCHEDULER / AMP

if USE_CLASS_WEIGHTED_LOSS:
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

print("Loss       :", criterion)
print("Optimizer  :", optimizer.__class__.__name__)
print("Scheduler  :", scheduler.__class__.__name__)
print("AMP enabled:", scaler.is_enabled())

In [ ]:
# CELL 13 : METRICS / TRAIN / EVAL FUNCTIONS

def topk_accuracy(logits, targets, k=3):
    k = min(k, logits.shape[1])
    topk = logits.topk(k, dim=1).indices
    correct = topk.eq(targets.unsqueeze(1)).any(dim=1)
    return correct.float().mean().item()

def run_epoch(model, loader, optimizer=None, training=False):
    model.train(training)

    losses = []
    all_true = []
    all_pred = []
    top3_sum = 0.0
    n_batches = 0

    for batch in loader:
        x = batch["x"].to(DEVICE, non_blocking=True)
        y = batch["y"].to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            "cuda",
            enabled=torch.cuda.is_available()
        ):
            logits = model(x)
            loss = criterion(logits, y)

        if training:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()

        pred = logits.argmax(dim=1)

        losses.append(loss.item())
        all_true.extend(y.detach().cpu().tolist())
        all_pred.extend(pred.detach().cpu().tolist())
        top3_sum += topk_accuracy(logits.detach(), y, k=3)
        n_batches += 1

    acc = accuracy_score(all_true, all_pred)
    macro_f1 = f1_score(all_true, all_pred, average="macro", zero_division=0)

    return {
        "loss": float(np.mean(losses)),
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "top3": float(top3_sum / max(n_batches, 1)),
        "y_true": all_true,
        "y_pred": all_pred,
    }

In [ ]:
# CELL 14 : SANITY OVERFIT TEST — MUST PASS BEFORE FULL TRAINING

# Small subset: 2 samples from up to 8 classes.
sanity_rows = (
    train_df
    .groupby("model_class_id", group_keys=False)
    .head(2)
    .head(16)
    .reset_index(drop=True)
)

sanity_ds = LipWordDataset(
    sanity_rows.assign(
        mouth_cache_path=sanity_rows.apply(lambda r: str(cache_path_for_row(r)), axis=1)
    ),
    train=False
)

sanity_loader = DataLoader(
    sanity_ds,
    batch_size=len(sanity_ds),
    shuffle=True,
    num_workers=0
)

sanity_model = MouthWordClassifier(num_classes=NUM_CLASSES).to(DEVICE)
sanity_optimizer = torch.optim.AdamW(sanity_model.parameters(), lr=1e-3)
sanity_criterion = nn.CrossEntropyLoss()

for epoch in range(1, 101):
    sanity_model.train()

    batch_s = next(iter(sanity_loader))
    x = batch_s["x"].to(DEVICE)
    y = batch_s["y"].to(DEVICE)

    sanity_optimizer.zero_grad(set_to_none=True)

    logits = sanity_model(x)
    loss = sanity_criterion(logits, y)
    loss.backward()
    sanity_optimizer.step()

    acc = (logits.argmax(1) == y).float().mean().item()

    if epoch == 1 or epoch % 10 == 0 or acc >= 0.99:
        print(f"epoch={epoch:03d} | loss={loss.item():.4f} | acc={acc:.3f}")

    if acc >= 0.99:
        print("\n✅ Sanity overfit passed")
        break
else:
    print("\n⚠️ Sanity overfit did not reach 99%. Investigate before full training.")

In [ ]:
# CELL 15 : FULL TRAINING

# Re-initialize main model to guarantee full training starts clean.
model = MouthWordClassifier(num_classes=NUM_CLASSES).to(DEVICE)

if USE_CLASS_WEIGHTED_LOSS:
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

history = []

best_f1 = -1.0
best_epoch = 0
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        training=True
    )

    with torch.no_grad():
        val_metrics = run_epoch(
            model,
            val_loader,
            optimizer=None,
            training=False
        )

    scheduler.step(val_metrics["macro_f1"])

    lr_now = optimizer.param_groups[0]["lr"]

    row = {
        "epoch": epoch,
        "lr": lr_now,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_f1": val_metrics["macro_f1"],
        "val_top3": val_metrics["top3"],
    }
    history.append(row)

    print(
        f"Epoch {epoch:03d} | "
        f"lr={lr_now:.2e} | "
        f"train loss={row['train_loss']:.4f} acc={row['train_acc']:.3f} f1={row['train_f1']:.3f} | "
        f"val loss={row['val_loss']:.4f} acc={row['val_acc']:.3f} "
        f"f1={row['val_f1']:.3f} top3={row['val_top3']:.3f}"
    )

    if val_metrics["macro_f1"] > best_f1:
        best_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        bad_epochs = 0

        checkpoint = {
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "best_val_macro_f1": best_f1,
            "num_classes": NUM_CLASSES,
            "target_frames": TARGET_FRAMES,
            "roi_size": ROI_SIZE,
            "architecture": "mouth_cnn_bigru_attention_baseline01",
            "label_map": label_map,
            "config": {
                "batch_size": BATCH_SIZE,
                "lr": LR,
                "weight_decay": WEIGHT_DECAY,
                "weighted_loss": USE_CLASS_WEIGHTED_LOSS,
                "weighted_sampler": USE_WEIGHTED_SAMPLER,
            },
        }

        torch.save(checkpoint, BEST_MODEL_PATH)
        print("  ✅ saved best:", BEST_MODEL_PATH)
    else:
        bad_epochs += 1

    if bad_epochs >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}; best epoch = {best_epoch}")
        break

history_df = pd.DataFrame(history)
history_path = MODEL_DIR / "01_training_history.csv"
history_df.to_csv(history_path, index=False, encoding="utf-8-sig")

print("\nBest epoch :", best_epoch)
print("Best val F1:", best_f1)
print("History    :", history_path)

In [ ]:
# CELL 16 : LOAD BEST MODEL + TEST EVALUATION

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

best_model = MouthWordClassifier(
    num_classes=checkpoint["num_classes"]
).to(DEVICE)

best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()

with torch.no_grad():
    test_metrics = run_epoch(
        best_model,
        test_loader,
        optimizer=None,
        training=False
    )

print("BEST CHECKPOINT")
print("epoch      :", checkpoint["epoch"])
print("val macroF1:", checkpoint["best_val_macro_f1"])

print("\nTEST")
print(f"loss      : {test_metrics['loss']:.4f}")
print(f"accuracy  : {test_metrics['accuracy']:.4f}")
print(f"macro F1  : {test_metrics['macro_f1']:.4f}")
print(f"top-3 acc : {test_metrics['top3']:.4f}")

In [ ]:
# CELL 17 : CLASSIFICATION REPORT

id_to_word_raw = label_map.get("id_to_word", {})
id_to_word = {int(k): v for k, v in id_to_word_raw.items()}

target_names = [
    id_to_word.get(i, f"class_{i}")
    for i in range(NUM_CLASSES)
]

report = classification_report(
    test_metrics["y_true"],
    test_metrics["y_pred"],
    labels=list(range(NUM_CLASSES)),
    target_names=target_names,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report).T

display(
    report_df.loc[target_names, ["precision", "recall", "f1-score", "support"]]
             .sort_values("f1-score")
)

In [ ]:
# CELL 18 : CONFUSION MATRIX

import matplotlib.pyplot as plt

cm = confusion_matrix(
    test_metrics["y_true"],
    test_metrics["y_pred"],
    labels=list(range(NUM_CLASSES))
)

fig = plt.figure(figsize=(16, 14))
ax = fig.add_subplot(111)

im = ax.imshow(cm, interpolation="nearest")

ax.set_title("V3 Mouth-only — Test Confusion Matrix")
ax.set_xlabel("Predicted class ID")
ax.set_ylabel("True class ID")

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))

plt.tight_layout()
plt.show()

In [ ]:
# CELL 19 : MOST CONFUSED CLASS PAIRS

pairs = []

for true_id in range(NUM_CLASSES):
    for pred_id in range(NUM_CLASSES):
        if true_id == pred_id:
            continue

        n = int(cm[true_id, pred_id])

        if n > 0:
            pairs.append({
                "true_id": true_id,
                "true_word": id_to_word.get(true_id, str(true_id)),
                "pred_id": pred_id,
                "pred_word": id_to_word.get(pred_id, str(pred_id)),
                "count": n,
            })

confused_df = pd.DataFrame(pairs)

if len(confused_df):
    display(confused_df.sort_values("count", ascending=False).head(30))
else:
    print("No off-diagonal errors.")

In [ ]:
# CELL 20 : INSPECT INDIVIDUAL PREDICTIONS

best_model.eval()

batch = next(iter(test_loader))
x = batch["x"].to(DEVICE)
y = batch["y"].to(DEVICE)

with torch.no_grad():
    logits = best_model(x)
    prob = torch.softmax(logits, dim=1)
    top_prob, top_idx = prob.topk(3, dim=1)

for i in range(min(12, len(y))):
    true_id = int(y[i].item())
    true_word = id_to_word.get(true_id, str(true_id))

    preds = []
    for j in range(3):
        pid = int(top_idx[i, j].item())
        pword = id_to_word.get(pid, str(pid))
        p = float(top_prob[i, j].item())
        preds.append(f"{pword} ({p:.3f})")

    print(
        f"[{i:02d}] TRUE={true_word:<12s} | "
        f"TOP3={preds} | "
        f"{batch['path'][i]}"
    )

In [ ]:
# CELL 21 : EXPORT FINAL V3 CONFIG SUMMARY

summary = {
    "project": "Thai visual word recognition",
    "version": "01 Supervised Word-Level Baseline",
    "task": "44-class word classification",
    "input": {
        "stream": "mouth-only",
        "frames": TARGET_FRAMES,
        "roi_size": ROI_SIZE,
        "channels": 3,
    },
    "model": "CNN frame encoder + BiGRU + temporal attention + classifier",
    "loss": "weighted CrossEntropyLoss" if USE_CLASS_WEIGHTED_LOSS else "CrossEntropyLoss",
    "ctc": False,
    "character_tokenizer": False,
    "best_model_path": str(BEST_MODEL_PATH),
}

summary_path = MODEL_DIR / "01_config_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nSaved:", summary_path)

## Next notebooks — after baseline is measured

Do **not** mix these changes into Notebook 01.

The planned sequence is:

1. **02 — Self-Supervised Visual Pretraining**
2. **03 — SSL Fine-Tuning on the 44-word dataset**
3. **04 — Temporal Modeling / Alignment Ablation**
4. **05 — Face + Mouth Dual-Stream Ablation**
5. **06 — LLM-Assisted Decoding / Top-k Rescoring**
6. **07 — Real-Time Inference / Prototype Evaluation**

Primary baseline question:

> Can mouth motion alone reliably classify the current 44-word Thai vocabulary?
